In [1]:
# [Cell 0] Install Dependencies
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.7 MB/s eta 0:00:00


In [2]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       940Mi       7.0Gi       2.0Mi       4.7Gi        11Gi
Swap:             0B          0B          0B


In [3]:
!nvidia-smi

Mon Sep  7 15:06:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
#[Cell 1] Environment Setup & Authentication
from google.colab import userdata
from huggingface_hub import login
import warnings

# Suppress the specific warning about missing tokens
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

try:
    # Attempt to fetch the token from Colab's secure storage
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.


In [5]:
#[Cell 2] Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load and configure the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load and configure the Neural Network Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
# [Cell 2b] Baseline Inference (Before Fine-Tuning)
messages = [{"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}]

# Format the prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
# print("Token IDs:\n", inputs["input_ids"][0][:20]) # Shows the first ~20 token IDs (numbers)

# Generate an answer using the untrained starting model
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract and print only the generated response
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("STARTING MODEL ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

STARTING MODEL ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information


In [7]:
# Automatically download the dataset if it hasn't been uploaded manually
!wget -nc -q https://raw.githubusercontent.com/AI-Learning-Repo/Data-Handling/refs/heads/week4/datasets/MediCore.json

In [8]:
#[Cell 3] Dataset Loading and Preprocessing
from datasets import load_dataset

raw_data = load_dataset("json", data_files="MediCore.json")

def preprocess(sample):
    messages = [
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]

    # Automatically applies <|im_start|> and <|im_end|> ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        #max_length=256,
        padding=False
    )
    # Explicitly create labels for loss calculation
    #tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(
    preprocess,
    remove_columns=raw_data["train"].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

In [9]:
# [Cell 4] LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

In [10]:
#[Cell 5] Training Setup and Execution
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# Let the collator handle padding + labels
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Split 10% of the data for validation
split = data["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=1,      # ↓ reduce to avoid OOM
    gradient_accumulation_steps=2,      # keeps effective batch size

    fp16=True,                          # ↓ big memory saver

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

# IMPORTANT: enable memory savings
model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.681174,0.639078
2,0.348880,0.704905
3,0.274709,0.796682
4,0.146847,0.906557
5,0.132602,1.010391


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

In [11]:
# [Cell 6] Load Model for Testing
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

# 1. Load the original starting-model checkpoint
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

# 2. Attach your tiny, fine-tuned adapter to the starting model
model = PeftModel.from_pretrained(base_model, path)

# 3. Re-enable caching for faster inference speeds
model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [12]:
# [Cell 7] Inference Execution
messages =[ {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"} ]

# Format the text with ChatML tags and generation prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert text to tensor numbers and move to GPU
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate the output
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Strip out the input prompt so we only see the newly generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("FINE-TUNED ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

FINE-TUNED ANSWER:
 Dr. Samuel Kaarlo leads the neurology department at MediCore Hospital.


In [14]:
# [Cell 10] Python-Enforced JSON

import json

messages = [
    {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

json_output = json.dumps(
    {"answer": response_text},
    indent=4
)

print(json_output)

{
    "answer": "Dr. Samuel Kaarlo leads the neurology department at MediCore Hospital."
}


In [15]:
from pydantic import BaseModel, Field

class HospitalResponse(BaseModel):
    answer: str = Field(
        description="The main text answer to the user's question"
    )
    model_version: str = Field(
        default="Qwen2.5-1.5B-MediCore",
        description="The model used"
    )

In [16]:
# [Cell 11] Pydantic Structured Output

from pydantic import BaseModel, Field

class HospitalResponse(BaseModel):
    answer: str = Field(
        description="The main text answer to the user's question"
    )
    model_version: str = Field(
        default="Qwen2.5-1.5B-MediCore",
        description="The model used"
    )

messages = [
    {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]
raw_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

structured_response = HospitalResponse(
    answer=raw_text
)

print(structured_response.model_dump_json(indent=4))

{
    "answer": "Dr. Samuel Kaarlo leads the neurology department at MediCore Hospital.",
    "model_version": "Qwen2.5-1.5B-MediCore"
}


In [17]:
# [Cell 12a]
# Install Streamlit and visualization libraries (fast pre-built binary wheels)
!pip install -q streamlit plotly seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.4 MB/s eta 0:00:00


In [20]:
# [Cell 12b] Streamlit Setup

# 1. Write baseline application
with open("app.py", "w") as f:
    f.write("""import streamlit as st
st.set_page_config(page_title="Streamlit Colab Lab", layout="centered")
st.title("Streamlit Environment Online")
st.success("Server initialized successfully. Proceed to Module 1 below.")
""")

# 2. Terminate any previous instances cleanly
import subprocess
import time
subprocess.run(["pkill", "-f", "streamlit"], stderr=subprocess.DEVNULL)

# 3. Launch Streamlit server in the background
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Allow server time to bind port
time.sleep(3)

# 4. Generate direct URL and display embedded iframe
from google.colab import output
from google.colab.output import eval_js

proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")

print("=====================================================================")
print(f"FULL-SCREEN DIRECT URL: {proxy_url}")
print("=====================================================================")

# Render interactive iframe in notebook cell output:
#output.serve_kernel_port_as_iframe(8501, height='650')

FULL-SCREEN DIRECT URL: https://8501-gpu-t4-s-kkb-usw4a0-3ot11v1bzn7ta-a.us-west4-0.prod.colab.dev


In [21]:
# [Cell 13] Streamlit GUI - Pydantic Structured Output

%%writefile app.py
import streamlit as st
import torch
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

st.set_page_config(
    page_title="MediCore Qwen Assistant",
    page_icon="🤖",
    layout="centered"
)

st.title("MediCore Fine-Tuned Qwen")
st.caption("Pydantic structured output")

class HospitalResponse(BaseModel):
    answer: str = Field(
        description="The main text answer to the user's question"
    )
    model_version: str = Field(
        default="Qwen2.5-1.5B-MediCore",
        description="The model used"
    )

@st.cache_resource
def load_model():
    path = "./my_qwen"

    config = PeftConfig.from_pretrained(path)

    tokenizer = AutoTokenizer.from_pretrained(
        config.base_model_name_or_path
    )
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name_or_path,
        device_map="cuda",
        torch_dtype=torch.float16
    )

    model = PeftModel.from_pretrained(base_model, path)
    model.config.use_cache = True

    return tokenizer, model

tokenizer, model = load_model()

user_prompt = st.text_area(
    "Enter your question",
    value="Who leads the neurology department at MediCore Hospital?",
    height=120
)

if st.button("Generate response"):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output[0][inputs.input_ids.shape[1]:]

    raw_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    structured_response = HospitalResponse(
        answer=raw_text
    )

    json_output = structured_response.model_dump_json(
        indent=4
    )

    st.subheader("Structured JSON Output")
    st.code(json_output, language="json")

Overwriting app.py
